# Supplier Queries

In [1]:
import pandas as pd
import duckdb

import ipywidgets as widgets
from ipywidgets import interact, Dropdown, IntRangeSlider

from juptils import show_pretty

\
Load monthly sales rollup data.  
<br />

In [2]:
monthly_sales = pd.read_csv("./csv/monthly-sales.csv")


## Yearly Sales by Supplier

In [3]:
def show_query(supplier_cond):

    sql_template = """
        SELECT 
            SUBSTR(month, 1, 4) AS year, 
            supplier AS supplier,
            format('{:t,}', SUM(month_amt)) AS ytd_amt
        FROM monthly_sales
        WHERE _SUPPLIER_SLUG_
        GROUP BY year, supplier
        ORDER BY year, supplier
        """.replace("_SUPPLIER_SLUG_", supplier_cond)
    
    df = duckdb.query(sql_template).df() 
    df.show_pretty(left=['supplier'])

   
interact(
    show_query, 
    supplier_cond=[
        ('all',         "1=1"),
        ('nippon-metal',"supplier = 'nippon-metal'"), 
        ('us-steel',    "supplier = 'us-steel'")]);

interactive(children=(Dropdown(description='supplier_cond', options=(('all', '1=1'), ('nippon-metal', "supplie…

# Quarterly Sales

In [11]:
# new
######

sql_template = """
    SELECT
      CAST(SUBSTR(month, 1, 4) AS INT64) AS year,
      CONCAT('Q', CAST(CEIL(CAST(SUBSTR(month, 6, 2) AS INT64) / 3.0) AS INT64)) AS qtr,
      supplier AS supplier,
      FORMAT('{:t,}', SUM(month_amt)) AS qtr_amt
      
    FROM monthly_sales
    
    WHERE _SUPPLIER_CONDITION_ 
        AND year BETWEEN _YEAR_MIN_ AND _YEAR_MAX_

    GROUP BY
      CAST(SUBSTR(month, 1, 4) AS INT64),
      CONCAT('Q', CAST(CEIL(CAST(SUBSTR(month, 6, 2) AS INT64) / 3.0) AS INT64)),
      supplier
      
    ORDER BY year, qtr, supplier;
"""

def show_query(supplier_cond, years):

    (year_min,year_max) = years

#    sql = sql_template.replace("_SUPPLIER_CONDITION_", supplier_cond).replace("_YEAR_MIN_", str(year_min)).replace("_YEAR_MAX_", str(year_max))
        
    sql = (
        sql_template
        .replace("_SUPPLIER_CONDITION_", supplier_cond)
        .replace("_YEAR_MIN_", str(year_min))
        .replace("_YEAR_MAX_", str(year_max))
    )
    
    df = duckdb.query(sql).df() 
    df.show_pretty(left=['supplier'])



suppliers = [('all',"1=1"),('nippon-metal',"supplier = 'nippon-metal'"), ('us-steel', "supplier = 'us-steel'")]

dropdown = Dropdown(options=suppliers, description='Supplier')
slider = IntRangeSlider(value=[2020,2030],min=2020, max=2030, step=1, description='Years')

interact(show_query, supplier_cond=dropdown, years=slider)

interactive(children=(Dropdown(description='Supplier', options=(('all', '1=1'), ('nippon-metal', "supplier = '…

<function __main__.show_query(supplier_cond, years)>